In [0]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("subscriptions").getOrCreate()

CUTOFF = "2024-01-01"

data = [
    (1, "2022-01-01", "2022-06-30", "Premium",  50.0),
    (1, "2022-07-15", "2023-01-15", "Premium",  50.0),
    (2, "2022-03-01", "2023-03-01", "Basic",    25.0),
    (3, "2023-01-01", "2024-06-30", "Standard", 35.0),
    (4, "2022-06-01", "2022-09-30", "Basic",    25.0),
]

subscriptions = spark.createDataFrame(
    data, ["customer_id", "start_date", "end_date", "plan", "monthly_fee"]
)

# --- Solution ---
subs = (
    subscriptions
    .withColumn("start_date", F.to_date("start_date", "yyyy-MM-dd"))
    .withColumn("end_date", F.to_date("end_date", "yyyy-MM-dd"))
    # months = ceil(days / 30)
    .withColumn("months", F.ceil(F.datediff("end_date", "start_date") / 30))
    .withColumn("sub_value", F.col("monthly_fee") * F.col("months"))
)

result = (
    subs.groupBy("customer_id")
    .agg(
        F.max("end_date").alias("last_end_date"),
        F.sum("sub_value").alias("lifetime_value"),
    )
    # churned = no subscription still active as of the cutoff
    .filter(F.col("last_end_date") < F.to_date(F.lit(CUTOFF)))
    .select(
        "customer_id",
        "last_end_date",
        F.round("lifetime_value", 2).alias("lifetime_value"),
    )
    .orderBy(F.col("lifetime_value").desc())
)

result.show()